# Steefel & MacQuarrie (1996) Figure 6 replication

This notebook visualizes the first-order decay benchmark generated by `run.py`. The solid black curve is the analytical semi-infinite advection–dispersion–decay solution. The model contains an explicit $x=0$ CNC boundary node; the reported numerical profiles are the following interior nodes, $x=j\Delta x$. A separate panel retains the raw MODFLOW DIS cell-center coordinates as a grid-coordinate sensitivity.

Generate or refresh the data first from the repository root:

```text
python examples/Splitting_KineticDecay/run.py
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 240,
        "font.size": 10.5,
        "axes.titleweight": "bold",
    }
)

candidates = [
    Path.cwd(),
    Path.cwd() / "examples" / "Splitting_KineticDecay",
]
CASE_DIR = next(
    (
        path.resolve()
        for path in candidates
        if (path / "output" / "paper_figure6_data.npz").exists()
    ),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError("Run examples/Splitting_KineticDecay/run.py first.")
OUTPUT_DIR = CASE_DIR / "output"
CASE_DIR

## Load and audit the simulation products

The archive contains only simulation output and the analytical reference; this notebook does not rerun MF6PQC.

In [ ]:
with np.load(OUTPUT_DIR / "paper_figure6_data.npz") as archive:
    arrays = {name: archive[name].copy() for name in archive.files}
metrics = pd.read_csv(OUTPUT_DIR / "paper_figure6_metrics.csv")

required = {
    "x_paper_nodes_m",
    "x_modflow_cell_centers_m",
    "x_analytical_dense_m",
    "analytical_dense",
    "analytical_paper_nodes",
    "analytical_cell_centers",
    "profile__SNIA__cfl_1",
    "profile__Strang__cfl_1",
    "profile__SIA__cfl_1",
}
missing = sorted(required - arrays.keys())
if missing:
    raise RuntimeError(f"Incomplete benchmark archive; missing: {missing}")
if metrics.empty or not np.isfinite(metrics["paper_node_rmse"]).all():
    raise RuntimeError("Metrics are empty or contain non-finite errors.")

display(
    metrics.sort_values(["method", "cfl"])[
        [
            "method",
            "cfl",
            "logical_steps",
            "damkohler_per_step",
            "paper_node_rmse",
            "cell_center_rmse",
            "boundary_endpoint_concentration",
            "transport_solves",
            "reaction_evaluations",
            "wall_time_seconds",
        ]
    ].reset_index(drop=True)
)

## Paper-style concentration profiles

The original Figure 6 shows SNIA and Strang at CFL = 0.1, 0.5, and 1, and states that SIA nearly overlays the analytical curve. The third panel makes that SIA comparison explicit.

MF6PQC now uses the paper's instantaneous endpoint-rate form for SIA and the MODFLOW CENTRAL stencil for this Pe=2 verification. SIA nearly overlays the analytical curve at every tested CFL; at CFL=1 the accuracy order is SIA, Strang, SNIA. SNIA approaches the reference only after its time step is reduced. Concentrations are normalized by the actual PhreeqcRM inlet value, so the comparison is C/C0.

In [ ]:
def cfl_token(cfl):
    return format(float(cfl), "g").replace(".", "p")


def profile_key(method, cfl):
    return f"profile__{method}__cfl_{cfl_token(cfl)}"


x_nodes = arrays["x_paper_nodes_m"]
x_dense = arrays["x_analytical_dense_m"]
analytical_dense = arrays["analytical_dense"]
styles = {
    0.1: dict(color="#4c78a8", linestyle=":", marker=None),
    0.5: dict(color="#f58518", linestyle="--", marker="o"),
    1.0: dict(color="#54a24b", linestyle="-.", marker="s"),
}

figure, axes = plt.subplots(1, 3, figsize=(14.2, 4.25), sharey=True, constrained_layout=True)
for axis, method in zip(axes[:2], ("SNIA", "Strang"), strict=False):
    axis.plot(x_dense, analytical_dense, color="black", lw=2.0, label="Analytical")
    for cfl in (0.1, 0.5, 1.0):
        key = profile_key(method, cfl)
        axis.plot(x_nodes, arrays[key], lw=1.6, ms=4.0, label=f"CFL = {cfl:g}", **styles[cfl])
    axis.set_title(method)

axis = axes[2]
axis.plot(x_dense, analytical_dense, color="black", lw=2.0, label="Analytical")
sia_rows = metrics.loc[metrics["method"].eq("SIA")].sort_values("cfl")
for cfl in sia_rows["cfl"]:
    key = profile_key("SIA", cfl)
    axis.plot(x_nodes, arrays[key], lw=1.7, ms=4.2, label=f"CFL = {cfl:g}", **styles[float(cfl)])
axis.set_title("SIA (explicit comparison)")

for axis in axes:
    axis.set(xlabel="Distance, x (m)", xlim=(0, 6), ylim=(0, 1.02))
    axis.legend(frameon=True, fontsize=9)
axes[0].set_ylabel(r"Normalized concentration, $C/C_0$")
figure.suptitle("First-order decay splitting benchmark at t = 0.5 yr", fontsize=13)
figure.savefig(OUTPUT_DIR / "steefel1996_figure6_replication.png", bbox_inches="tight")
plt.show()

## Error, coordinate sensitivity, and computational work

The left panel is the temporal/CFL study. The center panel prevents an accuracy claim from hiding its computational cost. The right panel compares the explicit conceptual node coordinates with the raw MODFLOW DIS cell centers; the latter is a grid-coordinate diagnostic, not the Figure-6 validation norm.

In [ ]:
method_colors = {"SNIA": "#4c78a8", "Strang": "#f58518", "SIA": "#54a24b"}
figure, axes = plt.subplots(1, 3, figsize=(14.5, 4.3), constrained_layout=True)

for method, group in metrics.groupby("method", sort=False):
    group = group.sort_values("cfl")
    axes[0].loglog(
        group["cfl"], group["paper_node_rmse"], "o-", color=method_colors[method], label=method
    )
axes[0].set(
    xlabel="CFL number",
    ylabel="RMSE (paper-node convention)",
    title="Error under coupling refinement",
)
axes[0].legend()

for method, group in metrics.groupby("method", sort=False):
    axes[1].loglog(
        group["transport_solves"],
        group["paper_node_rmse"],
        "o",
        ms=7,
        color=method_colors[method],
        label=method,
    )
    for row in group.itertuples():
        axes[1].annotate(
            f"{row.cfl:g}",
            (row.transport_solves, row.paper_node_rmse),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )
axes[1].set(xlabel="Transport solves", ylabel="RMSE", title="Accuracy–work trade-off")
axes[1].legend()

cfl_one = (
    metrics.loc[np.isclose(metrics["cfl"], 1.0)].set_index("method").loc[["SNIA", "Strang", "SIA"]]
)
positions = np.arange(3)
width = 0.36
axes[2].bar(positions - width / 2, cfl_one["paper_node_rmse"], width, label="Paper nodes")
axes[2].bar(positions + width / 2, cfl_one["cell_center_rmse"], width, label="Raw DIS centers")
axes[2].set_xticks(positions, cfl_one.index)
axes[2].set(ylabel="RMSE at CFL = 1", title="Grid-coordinate sensitivity")
axes[2].legend(fontsize=8.5)

for axis in axes:
    axis.grid(True, which="both", alpha=0.25)
figure.savefig(OUTPUT_DIR / "splitting_error_cost.png", bbox_inches="tight")
plt.show()

## Reproduction verdict

In [ ]:
paper_rank = cfl_one["paper_node_rmse"].sort_values()
center_rank = cfl_one["cell_center_rmse"].sort_values()
print("CFL=1 ranking with the paper-node convention:", " < ".join(paper_rank.index))
print("CFL=1 ranking with raw DIS centers:        ", " < ".join(center_rank.index))
print(
    f"SIA paper-node RMSE improvement over Strang: "
    f"{(1 - paper_rank['SIA'] / paper_rank['Strang']) * 100:.2f}%"
)
print(
    "Interpretation: the paper's inlet over-reaction signature is reproduced, "
    "but the method ranking must always be reported together with the spatial "
    "boundary convention and work count."
)